# 👟 ShoeBot – Train AI tư vấn giày (bấm 1 nút)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiensdattnek-ai/shoebot/blob/main/notebooks/colab_train_shoebot.ipynb)

**Cách dùng (3 thao tác):** `Runtime → Change runtime type → T4 GPU` → `Runtime → Run all` → chờ ~30–60 phút.

Notebook **tự tải code + dataset từ GitHub** — bạn không cần upload file nào cả.

🔗 Repo: https://github.com/tiensdattnek-ai/shoebot

| Bước | Việc | Thời gian |
|---|---|---|
| 1–2 | Kiểm tra GPU, tải code về Colab | ~1 phút |
| 3–4 | Cài thư viện, kiểm tra dữ liệu | ~2 phút |
| 5 | **Train** (Qwen2.5-3B · QLoRA 4-bit · 1.634 mẫu · 3 epoch) | ~30–60 phút |
| 6–8 | Hỏi thử, chấm điểm, gộp model, tải về | ~5 phút |

> Mẹo: muốn nhanh hơn chọn `MODEL = "Qwen/Qwen2.5-1.5B-Instruct"` ở ô cấu hình; muốn giọng hay hơn chọn `Qwen/Qwen2.5-7B-Instruct`.


## 1️⃣ Kiểm tra GPU
Nếu báo **KHÔNG CÓ GPU** → `Runtime → Change runtime type → T4 GPU` rồi chạy lại ô này.


In [ ]:
!nvidia-smi || echo "Chưa có GPU — hãy bật T4 trong Runtime → Change runtime type"
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")


## 2️⃣ Tải code từ GitHub (tự động)


In [ ]:
import os, subprocess
REPO_URL = "https://github.com/tiensdattnek-ai/shoebot.git"
PROJ = "/content/shoebot"

if not os.path.exists(PROJ):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJ], check=True)
else:
    subprocess.run(["git", "-C", PROJ, "pull", "--ff-only"], check=False)
os.chdir(PROJ)
print("✓ Đang ở:", os.getcwd())
!ls

# ---- tiện ích dùng cho các ô sau: chạy lệnh như trong terminal ----
import subprocess, sys, shlex

def run(*args):
    cmd = [str(a) for a in args]
    print("$", " ".join(shlex.quote(c) for c in cmd), "\n" + "-" * 70)
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {p.returncode}): {' '.join(cmd)}")
    print("-" * 70 + "  ✓ xong\n")


## 3️⃣ Cài thư viện


In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.34" bitsandbytes sentencepiece
import transformers, peft, datasets, torch
print("✓ transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__,
      "| CUDA", torch.cuda.is_available())


## 4️⃣ Cấu hình buổi train
Chỉ cần sửa ở ô dưới. Gợi ý: **1.5B** = nhanh nhất, **3B** = cân bằng ⭐, **7B** = hay nhất (cần Colab Pro/A100 hoặc chạy Unsloth).


In [ ]:
MODEL  = "Qwen/Qwen2.5-3B-Instruct"   # đổi thành "Qwen/Qwen2.5-1.5B-Instruct" nếu muốn nhanh hơn
EPOCHS = 3
BATCH  = 2
ACCUM  = 8
MAXLEN = 1024
OUT    = "out/shoebot-lora"
print(f"Model: {MODEL} | {EPOCHS} epoch | batch {BATCH} x {ACCUM} | max_len {MAXLEN} | adapter → {OUT}/final")


## 5️⃣ Kiểm tra dữ liệu + smoke test


In [ ]:
# sinh lại dataset từ knowledge base (đảm bảo mới nhất)
run(sys.executable, "data/build_dataset.py", "--out", "dataset")

# xem báo cáo
print(open("dataset/build_report.md", encoding="utf-8").read()[:1200])

# smoke test: kiểm tra toàn bộ pipeline trên CPU trước khi train thật
run(sys.executable, "train/smoke_test.py", "--steps", "4", "--subset", "8", "--max_len", "256")


## 6️⃣ TRAIN 🚀

Chạy xong là bạn có model riêng. Adapter lưu ở `out/shoebot-lora/final` (nhẹ, vài chục–trăm MB).
Trên T4: 3B mất khoảng 30–60 phút. **Đừng đóng tab trong lúc train** (Colab free sẽ ngắt phiên nếu tab ngủ).


In [ ]:
run(sys.executable, "train/train_lora.py",
    "--model", MODEL, "--load_in_4bit",
    "--data_dir", "dataset", "--output_dir", OUT,
    "--epochs", str(EPOCHS), "--batch_size", str(BATCH), "--grad_accum", str(ACCUM),
    "--max_len", str(MAXLEN), "--lora_r", "16", "--lora_alpha", "32", "--lr", "2e-4",
    "--logging_steps", "10")


## 7️⃣ Hỏi thử ngay 👇
Sửa `CAU_HOI` để hỏi câu của bạn.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, f"{OUT}/final")
model.eval()

SYSTEM = ("Bạn là ShoeBot – trợ lý bán hàng của một chuỗi bán lẻ giày/sneaker tại Việt Nam. "
          "Trả lời ngắn gọn, thân thiện, xưng 'em', gọi khách là 'anh/chị'.")

def ask(q):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    ids = enc["input_ids"] if hasattr(enc, "input_ids") else enc
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=300, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

CAU_HOI = [
    "Chân em 25cm, chân bè, đi Samba size gì ạ?",
    "So sánh Asics Gel-Kayano và Gel-Nimbus giúp em",
    "Ngân sách 2 triệu mua giày đi học thì chọn đôi nào?",
    "Cách check Nike Air Force 1 chính hãng?",
    "Giày da lộn dính bùn xử lý thế nào?",
]
for q in CAU_HOI:
    print("🧑", q)
    print("👟", ask(q))
    print("-" * 76)


## 8️⃣ Chấm điểm: model gốc vs model đã train


In [ ]:
print("=== (1) MODEL GỐC – chưa fine-tune ===")
run(sys.executable, "eval/evaluate.py", "--model", MODEL, "--limit", "12",
    "--questions", "eval/eval_questions.jsonl", "--out", "out/eval_goc.md")

print("=== (2) MODEL ĐÃ TRAIN ===")
run(sys.executable, "eval/evaluate.py", "--model", MODEL,
    "--adapter", f"{OUT}/final", "--load_in_4bit", "--limit", "12",
    "--questions", "eval/eval_questions.jsonl", "--out", "out/eval_finetuned.md")

import re
def bang(path, ten):
    print(f"\n--- {ten} ---")
    for l in open(path, encoding="utf-8"):
        if re.match(r"^\| \d+ \|", l):
            print(l.rstrip()[:150])
bang("out/eval_goc.md", "MODEL GỐC")
bang("out/eval_finetuned.md", "MODEL ĐÃ TRAIN")
print("\n(Đọc kỹ câu trả lời trong out/eval_finetuned.md — đó là sản phẩm thật của bạn)")


## 9️⃣ Gộp LoRA vào model (thành model hoàn chỉnh)


In [ ]:
run(sys.executable, "train/merge_adapter.py", "--model", MODEL,
    "--adapter", f"{OUT}/final", "--out", "out/shoebot-merged")
!ls -la out/shoebot-merged | head


## 🔟 Tải kết quả về máy


In [ ]:
import shutil
from google.colab import files

# (a) Adapter nhẹ (~50–200MB) — dùng ngay bằng --adapter khi chat
shutil.make_archive("/content/shoebot-lora", "zip", "out", "shoebot-lora")
files.download("/content/shoebot-lora.zip")

# (b) Model đã gộp (nặng hơn, chạy khỏi cần peft) — bỏ comment nếu muốn tải
# shutil.make_archive("/content/shoebot-merged", "zip", "out", "shoebot-merged")
# files.download("/content/shoebot-merged.zip")

# (c) Hoặc lưu vào Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r out/shoebot-lora/final "/content/drive/MyDrive/shoebot-lora-final"


## ✅ Xong! Dùng model trên máy bạn

```bash
git clone https://github.com/tiensdattnek-ai/shoebot.git && cd shoebot
pip install -r requirements.txt

# giải nén shoebot-lora.zip vào out/ rồi chat:
python app/chat_cli.py --model Qwen/Qwen2.5-3B-Instruct --adapter out/shoebot-lora/final

# hoặc dùng model đã gộp (nếu bạn có tải):
python app/chat_cli.py --model out/shoebot-merged

# giao diện web cho nhân viên:
pip install gradio
python app/app_gradio.py --model Qwen/Qwen2.5-3B-Instruct --adapter out/shoebot-lora/final

# KHÔNG CẦN GPU — mở app/shoebot_offline.html là tra cứu được 556 mục / 43 hãng
```

### Muốn nhanh gấp đôi + xuất GGUF chạy Ollama?
```python
!pip -q install unsloth
!python train/train_unsloth.py --model unsloth/Qwen2.5-3B-Instruct --epochs 3 --export_gguf
```

### Muốn bot "chuẩn shop" hơn?
Sửa `data/kb/support.py` (chính sách bảo hành/đổi trả thật) và giá trong `data/kb/brands_*.py`
→ chạy lại ô số 5 → train lại. Không cần sửa code.

*Repo: https://github.com/tiensdattnek-ai/shoebot*
